In [ ]:
import pandas as pd
import polars as pl
import numpy as np
import os
from pathlib import Path
import pandas as pd
import re, pathlib

In [ ]:
# =============================================================================
# 1.  Directory layout – pathlib all the way
# =============================================================================
SCRIPT_DIR   = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
PROJECT_ROOT = SCRIPT_DIR.parent          # edit if your notebook is elsewhere

DATA_DIR       = PROJECT_ROOT / "data/"
SIMULATION_DIR = DATA_DIR / "simulations/"          # folder with ATTRIBUTE_* and wide SSP file
TORNADO_SIM_DIR = SIMULATION_DIR /"data_for_LSU"
OUTPUT_DIR     = DATA_DIR / "output/"

In [ ]:
louisiana = pd.read_csv(TORNADO_SIM_DIR / "louisiana.csv")

In [ ]:
louisiana

In [ ]:
# 1) Filter out the base case
base_case = louisiana[louisiana['primary_id'] == 0].copy()


In [ ]:
keywords = ['demand', 'trns']
[col for col in louisiana.columns if all(k in col for k in keywords)]

In [ ]:
[col for col in louisiana.columns if "vehicle_distance_traveled_trns_road_light_electricity" in col]

In [ ]:
[col for col in louisiana.columns if stratswith(k in col for k in keywords)]

In [ ]:
# 2) Define your fuels
relevant_fuels = [
    'biomass', 'coal', 'coke', 'diesel', 'electricity',
    'furnace_gas', 'gasoline', 'hydrocarbon_gas_liquids',
    'hydrogen', 'kerosene', 'natural_gas', 'oil'
]


In [ ]:
# 3) Initialize accumulators
total_fuel_consumption_avoided_by_efficiency = pd.Series(0.0, index=base_case.index)
total_fuel_demand = pd.DataFrame({'time_period': base_case['time_period']}, index=base_case.index)

In [ ]:
# 4) Loop over fuels 
for fuel in relevant_fuels:
    # find the efficiency column(s) for this fuel
    eff_cols = [c for c in base_case.columns
                if c.startswith(f'efficfactor_enfu_industrial_energy_fuel_{fuel}')]
    # find the demand column(s) for this fuel
    dem_cols = [c for c in base_case.columns
                if (fuel in c and 'energy_demand_enfu_subsector_total_pj_inen' in c)]
    if not eff_cols or not dem_cols:
        continue

    fuel_efficiency = base_case[eff_cols[0]]
    fuel_demand     = base_case[dem_cols[0]]

    # store demand
    total_fuel_demand[fuel] = fuel_demand

    # calculate avoided consumption
    fuel_consumed          = fuel_demand / fuel_efficiency
    baseline_efficiency    = fuel_efficiency.iloc[0]
    fuel_consumed_baseline = fuel_demand / baseline_efficiency
    delta_consumption      = fuel_consumed_baseline - fuel_consumed

    total_fuel_consumption_avoided_by_efficiency += delta_consumption

In [ ]:
# 5) Build the output DataFrame
output_data = pd.DataFrame({
    'time_period': base_case['time_period'],
    'efficiency_energy_saving_in_PJ': total_fuel_consumption_avoided_by_efficiency
}, index=base_case.index)

In [ ]:
# 6) Apply your CAPEX/OPEX multipliers
capex_multiplier = 10_000_000
output_data['efficiency_capex'] = output_data['efficiency_energy_saving_in_PJ'] * capex_multiplier
output_data['efficiency_opex']  = 0

In [ ]:
# 7) Industrial cost parameters
capex_industrial_electricity = 92666.6 * 21
capex_industrial_other       = 92666.6 * 12
opex_industrial_electricity  = 92666.6 * 2.5
opex_industrial_other        = 92666.6 * 4.5

In [ ]:

# 8) Demand & cost breakdown
output_data['industrial_energy_demand_for_electricity_in_PJ'] = total_fuel_demand['electricity']
output_data['industrial_energy_demand_for_other_fuels_in_PJ'] = (
    total_fuel_demand[relevant_fuels].sum(axis=1) - total_fuel_demand['electricity']
)

output_data['electricity_capex'] = (
    capex_industrial_electricity * output_data['industrial_energy_demand_for_electricity_in_PJ']
)
output_data['other_fuel_capex'] = (
    capex_industrial_other * output_data['industrial_energy_demand_for_other_fuels_in_PJ']
)
output_data['electricity_opex'] = (
    opex_industrial_electricity * output_data['industrial_energy_demand_for_electricity_in_PJ']
)
output_data['other_fuel_opex'] = (
    opex_industrial_other * output_data['industrial_energy_demand_for_other_fuels_in_PJ']
)

In [ ]:


# 9) Write out to CSV

output_data.to_csv(OUTPUT_DIR/'industrial_energy_cost.csv', index=False)
